# Name That Model - Training Pipeline

This notebook trains a model to identify the authorship of AI-generated text passages among 18 candidate frontier language models.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

Libraries loaded


## 1. Load Data

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f'Training samples: {len(train)}')
print(f'Test samples: {len(test)}')
print(f'Number of classes: {train["model"].nunique()}')

Training samples: 2699
Test samples: 3
Number of classes: 18


## 2. Train Classifier

In [3]:
# Encode labels
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(train['model'])

# Use character and word n-grams for better stylistic capture
char_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(2, 5),
    max_features=10000,
    lowercase=True
)

word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 3),
    max_features=8000,
    stop_words='english',
    lowercase=True
)

# Combine features
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer

pipeline = Pipeline([
    ('tfidf', FeatureUnion([
        ('char', char_vectorizer),
        ('word', word_vectorizer)
    ])),
    ('clf', LogisticRegression(
        max_iter=2000,
        solver='lbfgs',
        C=2.0,
        random_state=42
    ))
])

print('Training classifier...')
pipeline.fit(train['text'].astype(str), encoded_labels)
print('Training complete')

Training classifier...


Training complete


## 3. Evaluate

In [4]:
train_preds = pipeline.predict(train['text'].astype(str))
bal_acc = balanced_accuracy_score(encoded_labels, train_preds)
print(f'Balanced accuracy: {bal_acc:.4f}')

Balanced accuracy: 0.9996


## 4. Predict and Create Submission

In [5]:
# Predict on test data
test_preds = pipeline.predict(test['text'].astype(str))
test_labels = label_encoder.inverse_transform(test_preds)

# Create submission dataframe
submission = pd.DataFrame({
    'id': test['id'].astype(str),
    'model': test_labels
})

# Ensure model names are valid
valid_models = [
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'
]
submission['model'] = submission['model'].apply(lambda x: x if x in valid_models else 'gpt-5.5')

# Ensure no null values and no duplicates in id
submission = submission.drop_duplicates(subset=['id'], keep='first')
submission = submission.dropna(subset=['id', 'model'])

# Save submission as CSV with header id,model
submission.to_csv('submission.csv', index=False)

print('Submission created: submission.csv')
print(submission.to_string(index=False))

Submission created: submission.csv
          id            model
00353fb1c541  claude-opus-4-6
005f5a9a6139    gpt-5.6-terra
0062a53d5487 gemini-3.7-flash
